# 19 · Gray-Box AIA72 Applicability / Shift Audit

Post-hoc Phase C1 audit using **currently retained per-case signals only**. No fitting, recalibration, threshold tuning, policy selection, or redesign.

The earlier SHARP policy experiment documented a squared Mahalanobis feature-distance OOD gate (Ledoit–Wolf fit on 2010–2013 transformed 3×16 SHARP histories; frozen q99 cutoff 379.55586300796773). However, the raw tensor / transform and saved per-case distance artifacts are no longer present on the current VM, Mac, connected Drive, or current GCS bucket. Therefore this notebook **does not reconstruct or fabricate SHARP feature-space distance**. It instead audits the retained applicability proxies: input status, seed spread, SHARP–AIA disagreement, fusion entropy, conformal failure, and their association with forecast errors across Cycle 24 → Cycle 25 → 2026.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp, spearmanr

ROOT=Path('/home/abmoses2000')
SEEDS=[17,29,43]

MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005'/'graybox_aia72_master_predictions.csv.gz'
MASTER_SHA='17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
SHARP_SHA='7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
MOND=ROOT/'graybox_aia72_conformal_mondrian_v2_20261005'/'mondrian_thresholds.csv'
ALARM=ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005'/'alarm_thresholds.csv'

OUT=ROOT/'graybox_aia72_applicability_shift_v1_20261006'
if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
(OUT/'figures').mkdir(parents=True)

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for c in iter(lambda:f.read(4*1024*1024),b''):
            h.update(c)
    return h.hexdigest()

def require(cond,msg):
    if not cond:
        raise ValueError(msg)

for p in [MASTER,SHARP,MOND,ALARM]:
    require(p.is_file(), f'Missing frozen input: {p}')
require(sha256(MASTER)==MASTER_SHA,'Master SHA changed')
require(sha256(SHARP)==SHARP_SHA,'SHARP SHA changed')

# ------------------------------------------------------------
# Load and align frozen predictions
# ------------------------------------------------------------
a=pd.read_csv(MASTER,low_memory=False)
s=pd.read_csv(SHARP,low_memory=False)
a['forecast_case_id']=a['forecast_case_id'].astype(str)
s['forecast_case_id']=s['forecast_case_id'].astype(str)
require(a['forecast_case_id'].is_unique and s['forecast_case_id'].is_unique,'Duplicate forecast_case_id')
m=a.merge(s,on='forecast_case_id',how='inner',suffixes=('_aia','_sharp'),validate='one_to_one')
require(len(m)==len(a),f'Alignment lost rows: {len(a)} -> {len(m)}')

label_a='label_aia' if 'label_aia' in m.columns else 'label'
label_s='label_sharp' if 'label_sharp' in m.columns else label_a
m['label_final']=pd.to_numeric(m[label_a],errors='raise').astype(int)
require(np.array_equal(m['label_final'].to_numpy(),pd.to_numeric(m[label_s],errors='raise').astype(int).to_numpy()),'AIA/SHARP labels disagree')

# Robust issue-time recovery
issue=pd.Series(pd.NaT,index=m.index,dtype='datetime64[ns, UTC]')
for c in [x for x in ['issue_utc_aia','issue_utc_sharp','issue_utc','T_REC_dt_aia','T_REC_dt_sharp','T_REC_dt','T_REC_aia','T_REC_sharp','T_REC'] if x in m.columns]:
    z=pd.to_datetime(m[c],utc=True,errors='coerce')
    take=issue.isna()&z.notna()
    issue.loc[take]=z.loc[take]
id_time=m['forecast_case_id'].str.extract(r':(\d{8}_\d{4})_',expand=False)
z=pd.to_datetime(id_time,format='%Y%m%d_%H%M',utc=True,errors='coerce')
take=issue.isna()&z.notna()
issue.loc[take]=z.loc[take]
require(issue.notna().all(),'Unresolved issue times')
m['issue_dt']=issue
m['year']=issue.dt.year.astype(int)
role_col='role_aia' if 'role_aia' in m.columns else 'role'
m['role_final']=m[role_col].astype(str)

m['sharp_probability']=pd.to_numeric(m['probability_gru_mean'],errors='coerce')
m['aia_probability']=pd.to_numeric(m['probability'],errors='coerce')
m['fusion_probability']=0.5*m['sharp_probability']+0.5*m['aia_probability']

def seed_cols(side):
    out=[]
    for z in SEEDS:
        if f'probability_seed_{z}_{side}' in m.columns:
            out.append(f'probability_seed_{z}_{side}')
        elif f'probability_seed_{z}' in m.columns:
            out.append(f'probability_seed_{z}')
        else:
            raise KeyError(f'Missing seed {z} for {side}')
    return out

aia_seed=m[seed_cols('aia')].astype(float).to_numpy()
sharp_seed=m[seed_cols('sharp')].astype(float).to_numpy()
fusion_seed=0.5*(aia_seed+sharp_seed)

m['aia_seed_std']=np.nanstd(aia_seed,axis=1,ddof=1)
m['sharp_seed_std']=np.nanstd(sharp_seed,axis=1,ddof=1)
m['fusion_seed_std']=np.nanstd(fusion_seed,axis=1,ddof=1)
m['sharp_aia_probability_gap']=np.abs(m['sharp_probability']-m['aia_probability'])

q=np.clip(m['fusion_probability'].to_numpy(float),1e-12,1-1e-12)
m['fusion_entropy']=-(q*np.log(q)+(1-q)*np.log(1-q))

def regime(row):
    if row['role_final']=='retrospective_cycle25':
        return 'cycle25_2021_2025_diagnostic'
    if row['role_final']=='supplementary_2026':
        return 'supplementary_2026_locked'
    if row['year']<=2019:
        return 'cycle24_earlier_development'
    return 'other_or_gap'
m['regime']=m.apply(regime,axis=1)

# ------------------------------------------------------------
# Input quality and conformal coverage
# ------------------------------------------------------------
aia_status_col='input_status' if 'input_status' in m.columns else ('input_status_aia' if 'input_status_aia' in m.columns else None)
if aia_status_col is None:
    m['aia_quality_ok']=True
else:
    m['aia_quality_ok']=m[aia_status_col].astype(str).eq('ok')

m['sharp_quality_ok']=np.isfinite(m['sharp_probability'])

mond=pd.read_csv(MOND)
mq={}
for b in ['sharp','aia','sharp_aia']:
    z=mond[(mond['branch']==b)&np.isclose(mond['alpha'].astype(float),0.10)]
    require(len(z)==1,f'Missing Mondrian threshold for {b}')
    mq[b]=(float(z.iloc[0]['qhat_class0']),float(z.iloc[0]['qhat_class1']))

def coverage(p,y,q0,q1):
    p=np.asarray(p,float)
    y=np.asarray(y,int)
    finite=np.isfinite(p)
    include0=finite&(p<=q0)
    include1=finite&((1-p)<=q1)
    return np.where(y==1,include1,include0)

y=m['label_final'].to_numpy(int)
for b,pcol in [('sharp','sharp_probability'),('aia','aia_probability'),('sharp_aia','fusion_probability')]:
    m[f'{b}_covered']=coverage(m[pcol].to_numpy(float),y,*mq[b])

alarm=pd.read_csv(ALARM)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}
for b,pcol in [('sharp','sharp_probability'),('aia','aia_probability'),('sharp_aia','fusion_probability')]:
    pred=(m[pcol].to_numpy(float)>=alarm_map[b]).astype(int)
    m[f'{b}_alarm_error']=pred!=y

# ------------------------------------------------------------
# 1) Regime-level applicability summary
# ------------------------------------------------------------
rows=[]
for rg,g in m.groupby('regime',sort=False):
    flare=g['label_final'].eq(1)
    rows.append({
        'regime':rg,
        'cases':len(g),
        'positives':int(flare.sum()),
        'prevalence':float(flare.mean()),
        'aia_quality_failure_rate':float((~g['aia_quality_ok']).mean()),
        'sharp_quality_failure_rate':float((~g['sharp_quality_ok']).mean()),
        'aia_seed_std_mean':float(g['aia_seed_std'].mean()),
        'sharp_seed_std_mean':float(g['sharp_seed_std'].mean()),
        'fusion_seed_std_mean':float(g['fusion_seed_std'].mean()),
        'sharp_aia_gap_mean':float(g['sharp_aia_probability_gap'].mean()),
        'fusion_entropy_mean':float(g['fusion_entropy'].mean()),
        'sharp_error_rate':float(g['sharp_alarm_error'].mean()),
        'aia_error_rate':float(g['aia_alarm_error'].mean()),
        'fusion_error_rate':float(g['sharp_aia_alarm_error'].mean()),
        'sharp_flare_coverage':float(g.loc[flare,'sharp_covered'].mean()) if flare.any() else np.nan,
        'aia_flare_coverage':float(g.loc[flare,'aia_covered'].mean()) if flare.any() else np.nan,
        'fusion_flare_coverage':float(g.loc[flare,'sharp_aia_covered'].mean()) if flare.any() else np.nan,
    })
reg=pd.DataFrame(rows)
reg.to_csv(OUT/'regime_applicability_summary.csv',index=False)

# ------------------------------------------------------------
# 2) Association of retained signals with failure
# ------------------------------------------------------------
signals={
    'sharp_aia_probability_gap':'sharp_aia_probability_gap',
    'fusion_seed_std':'fusion_seed_std',
    'sharp_seed_std':'sharp_seed_std',
    'aia_seed_std':'aia_seed_std',
    'fusion_entropy':'fusion_entropy',
}
m['sharp_conformal_failure']=~m['sharp_covered'].astype(bool)
m['aia_conformal_failure']=~m['aia_covered'].astype(bool)
m['fusion_conformal_failure']=~m['sharp_aia_covered'].astype(bool)

targets={
    'sharp_alarm_error':'sharp_alarm_error',
    'aia_alarm_error':'aia_alarm_error',
    'fusion_alarm_error':'sharp_aia_alarm_error',
    'sharp_conformal_failure':'sharp_conformal_failure',
    'aia_conformal_failure':'aia_conformal_failure',
    'fusion_conformal_failure':'fusion_conformal_failure',
}

assoc=[]
for rg,g in m.groupby('regime',sort=False):
    for sname,scol in signals.items():
        x=g[scol].to_numpy(float)
        finite=np.isfinite(x)
        for tname,tcol in targets.items():
            t=g[tcol].astype(int).to_numpy()
            use=finite&np.isfinite(t)
            auc=roc_auc_score(t[use],x[use]) if len(np.unique(t[use]))==2 else np.nan
            rho,pv=spearmanr(x[use],t[use]) if use.sum()>2 else (np.nan,np.nan)
            assoc.append({
                'regime':rg,'signal':sname,'target':tname,'cases':int(use.sum()),
                'roc_auc_for_failure':float(auc) if np.isfinite(auc) else np.nan,
                'spearman_rho':float(rho) if np.isfinite(rho) else np.nan,
                'spearman_p':float(pv) if np.isfinite(pv) else np.nan,
            })
assoc=pd.DataFrame(assoc)
assoc.to_csv(OUT/'signal_failure_associations.csv',index=False)

# ------------------------------------------------------------
# 3) Distribution shift of retained signals vs earlier Cycle 24
# ------------------------------------------------------------
base=m[m['regime'].eq('cycle24_earlier_development')]
shift=[]
for rg in ['cycle25_2021_2025_diagnostic','supplementary_2026_locked']:
    cur=m[m['regime'].eq(rg)]
    for sname,scol in signals.items():
        x0=base[scol].to_numpy(float)
        x1=cur[scol].to_numpy(float)
        stat,pv=ks_2samp(x0,x1,alternative='two-sided',method='auto')
        shift.append({
            'reference':'cycle24_earlier_development',
            'comparison':rg,
            'signal':sname,
            'ks_statistic':float(stat),
            'ks_p':float(pv),
            'reference_mean':float(np.mean(x0)),
            'comparison_mean':float(np.mean(x1)),
            'reference_median':float(np.median(x0)),
            'comparison_median':float(np.median(x1)),
        })
shift=pd.DataFrame(shift)
shift.to_csv(OUT/'distribution_shift_tests.csv',index=False)

# ------------------------------------------------------------
# 4) Error/coverage stratification by retained signal quantiles
# Quantile edges come from earlier Cycle-24 support and are descriptive only.
# ------------------------------------------------------------
strat=[]
for sname,scol in signals.items():
    ref=base[scol].to_numpy(float)
    edges=np.unique(np.quantile(ref,[0,.5,.75,.9,.95,.99,1.0]))
    if len(edges)<3:
        continue
    bins=np.r_[-np.inf,edges[1:-1],np.inf]
    temp=m[['regime','label_final',scol,'sharp_alarm_error','aia_alarm_error','sharp_aia_alarm_error','sharp_covered','aia_covered','sharp_aia_covered']].copy()
    temp['signal_bin']=pd.cut(temp[scol],bins=bins,include_lowest=True,duplicates='drop')
    for rg,g in temp.groupby('regime',sort=False):
        for b,h in g.groupby('signal_bin',observed=True):
            flare=h['label_final'].eq(1)
            strat.append({
                'signal':sname,'regime':rg,'signal_bin':str(b),'cases':len(h),'positives':int(flare.sum()),
                'signal_mean':float(h[scol].mean()),
                'sharp_error_rate':float(h['sharp_alarm_error'].mean()),
                'aia_error_rate':float(h['aia_alarm_error'].mean()),
                'fusion_error_rate':float(h['sharp_aia_alarm_error'].mean()),
                'sharp_flare_coverage':float(h.loc[flare,'sharp_covered'].mean()) if flare.any() else np.nan,
                'aia_flare_coverage':float(h.loc[flare,'aia_covered'].mean()) if flare.any() else np.nan,
                'fusion_flare_coverage':float(h.loc[flare,'sharp_aia_covered'].mean()) if flare.any() else np.nan,
            })
pd.DataFrame(strat).to_csv(OUT/'signal_stratified_diagnostics.csv',index=False)

# ------------------------------------------------------------
# Figures
# ------------------------------------------------------------
order=['cycle24_earlier_development','cycle25_2021_2025_diagnostic','supplementary_2026_locked']
pr=reg.set_index('regime').loc[[x for x in order if x in reg['regime'].values]]

for cols,title,name in [
    (['sharp_aia_gap_mean','fusion_seed_std_mean','fusion_entropy_mean'],'Retained agreement/confidence signals across regimes','fig1_trust_signals_by_regime.png'),
    (['sharp_error_rate','aia_error_rate','fusion_error_rate'],'Alarm error rate across regimes','fig2_error_rate_by_regime.png'),
    (['sharp_flare_coverage','aia_flare_coverage','fusion_flare_coverage'],'Flare-class conformal coverage across regimes','fig3_flare_coverage_by_regime.png'),
]:
    ax=pr[cols].plot(kind='bar',figsize=(10,5))
    ax.set_title(title); ax.set_xlabel('')
    if 'coverage' in name:
        ax.axhline(.90,linestyle='--',linewidth=1)
        ax.set_ylim(0,1)
    plt.xticks(rotation=15,ha='right')
    plt.tight_layout()
    plt.savefig(OUT/'figures'/name,dpi=220)
    plt.close()

# ------------------------------------------------------------
# Protocol
# ------------------------------------------------------------
protocol={
    'status':'POSTHOC_APPLICABILITY_SHIFT_AUDIT_COMPLETE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'horizon_hours':72,
    'new_model_fitting':False,
    'new_probability_recalibration':False,
    'new_threshold_tuning':False,
    'new_policy_selection':False,
    'new_policy_redesign':False,
    'retained_signals':['aia/sharp/fusion seed spread','SHARP-AIA probability gap','fusion entropy','input quality','conformal coverage/failure'],
    'sharp_feature_distance_status':'historically documented but not recomputed because raw tensor/transform and per-case distance artifacts are not retained in current execution environment',
    'historical_sharp_feature_distance_method':'squared Mahalanobis distance with Ledoit-Wolf covariance fitted on 2010-2013 transformed 3x16 SHARP histories',
    'historical_sharp_q99_threshold':379.55586300796773,
    'aia_feature_space_ood_status':'not available; no image-embedding OOD claim',
    'evidence_boundary':{
        'cycle24':'development/descriptive',
        'cycle25':'diagnostic/post-hoc for v3',
        '2026':'already-spent locked evidence; no redesign',
    },
    'source_sha256':{'master_predictions':MASTER_SHA,'sharp_predictions':SHARP_SHA},
    'outputs':['regime_applicability_summary.csv','signal_failure_associations.csv','distribution_shift_tests.csv','signal_stratified_diagnostics.csv','figures/'],
}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== APPLICABILITY / SHIFT AUDIT COMPLETE =====')
print(reg.round(6).to_string(index=False))
print('\nDistribution shifts:')
print(shift.round(6).to_string(index=False))
print('\nOutputs:',OUT)


===== APPLICABILITY / SHIFT AUDIT COMPLETE =====
                      regime  cases  positives  prevalence  aia_quality_failure_rate  sharp_quality_failure_rate  aia_seed_std_mean  sharp_seed_std_mean  fusion_seed_std_mean  sharp_aia_gap_mean  fusion_entropy_mean  sharp_error_rate  aia_error_rate  fusion_error_rate  sharp_flare_coverage  aia_flare_coverage  fusion_flare_coverage
 cycle24_earlier_development  24046       1406    0.058471                  0.162397                         0.0           0.045246             0.005886              0.023227            0.073463             0.197988          0.117442        0.551901           0.196665              0.805832            0.811522               0.837838
cycle25_2021_2025_diagnostic  35846       3860    0.107683                  0.000000                         0.0           0.030113             0.008914              0.017275            0.079875             0.198457          0.196424        0.387826           0.203788              0

## Interpretation guardrails

- These are applicability **proxies**, not a replacement for the missing per-case SHARP feature-distance artifact.
- The earlier SHARP Mahalanobis OOD method remains documented historical evidence, but this run does not reconstruct it.
- Signal/error associations are descriptive and post-hoc; they must not be used to retune v3 on Cycle 25 or 2026.
- AIA seed spread is not an image-embedding OOD detector.
- If disagreement/spread/entropy decrease while error and flare-coverage failure increase, that supports the conclusion that predictive agreement alone is insufficient for operational trust.
